# 🦟 Detekuje malárii lépe člověk, nebo notebook?

### Miniprojekt — Veletrh vědy · Katedra matematiky FJFI ČVUT

Malárie každý rok zabije přibližně **600 000 lidí**. Drtivá většina obětí žije v oblastech,
kde chybí vyškolení patologové, kteří by uměli pod mikroskopem rozpoznat parazita
*Plasmodium* v krevním nátěru. Diagnóza přitom stojí na jediné otázce, opakované u tisíců
buněk: **je tahle červená krvinka napadená, nebo zdravá?**

Co kdyby tu práci zvládl obyčejný mobil s dobře natrénovaným algoritmem?

Dnes si to vyzkoušíme. Použijeme veřejný dataset amerického Národního institutu zdraví (NIH)
s **27 558 mikroskopickými snímky** jednotlivých buněk — polovina zdravých, polovina
napadených.

---

### Co je hluboké učení a *transfer learning*?

**Neuronová síť** je matematická funkce s miliony nastavitelných čísel (*vah*), kterou
„naučíme" tím, že jí ukazujeme příklady a opravujeme její chyby. **Konvoluční** neuronová síť
je typ, který se specializuje na obrázky — postupně z pixelů skládá hrany, textury, tvary.

Natrénovat takovou síť od nuly vyžaduje miliony obrázků a hodiny počítání na výkonných
grafických kartách. My na to nemáme čas ani data — a nemusíme. Použijeme **transfer learning**:

> Vezmeme síť **ResNet**, kterou už někdo natrénoval na milionech běžných fotek
> (psi, auta, židle…), a využijeme to, že se naučila *vidět* — rozpoznávat tvary a textury.
> Tuhle „zrakovinu vidění" necháme **zmraženou** a dotrénujeme jen malou **klasifikační
> hlavu**, která se rozhodne: *infikovaná, nebo zdravá?*

To je celý dnešní trik. A protože je zbytek sítě zmražený, trénink poběží i na vašem
notebooku **bez grafické karty** — během pár sekund.

---

### Co dnes uděláte

1. Podíváte se na skutečné snímky buněk.
2. Pochopíte, jak ResNet promění obrázek na **2048 čísel** (tzv. *featury*).
3. **Navrhnete a natrénujete vlastní klasifikační hlavu.**
4. Naučíte se, proč v medicíně **nestačí hlásit „95 % přesnost"** — a co je *senzitivita*,
   *specificita* a *matice záměn*.
5. **Soutěž!** Dva týmy proti sobě. Vyhrává model, který nejlépe zachytí nemocné, aniž by
   příliš často zbytečně strašil zdravé.

Vzhůru do toho. 🚀

## 0 · Jak tenhle notebook ovládat

- Notebook se skládá z **buněk**. Buňku spustíte klávesou **`Shift + Enter`**.
- Buňky spouštějte **odshora dolů** — pozdější počítají s výsledky dřívějších.
- Nepotřebujete grafickou kartu. (Pokud chcete, v menu *Runtime → Change runtime type* můžete
  zapnout GPU, ale není to nutné.)
- Místa označená **`# 🔬 EXPERIMENT`** jsou vaše hřiště — tam měňte hodnoty a zkoušejte, jak
  se model zlepší. Zbytek nechte běžet, jak je.

➡️ **Nejdřív vyplňte jméno svého týmu v další buňce.**

In [ ]:
# ====== NASTAVENÍ ======
TEAM_NAME = "TYM_A"          # 🔬 napiš sem název svého týmu (např. "Plasmodium_Hunters")

# Odkud se stáhnou předpočítaná data (GitHub Release).
# Pokud nejsou dostupná (privátní repo), notebook si data obstará sám z veřejného NIH zdroje.
DATA_BASE_URL = "https://github.com/michalprusek/malaria/releases/download/data-v1"
# =======================

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Pro reprodukovatelnost (stejný start → stejné výsledky)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| zařízení:", device)
print("Tým:", TEAM_NAME)

## 1 · Získání dat

Notebook si data obstará automaticky a sám pozná, kterou ze dvou cest jít:

- **Rychlá cesta** — pokud školitel zveřejnil předpočítané featury, stáhnou se hotové
  soubory `train.npz`, `val.npz`, `test_features.npz` (pár sekund, GPU netřeba).
- **Soběstačná cesta** — když předpočítaná data nejsou veřejně dostupná, notebook si stáhne
  původní obrázky z veřejného serveru **NIH** a featury si **sám jednou spočítá** ResNetem.
  Trvá to ~5 minut a potřebuje zapnuté GPU: *Runtime → Change runtime type → GPU*.
  (Bonus: uvidíte ResNet skutečně běžet na obrázcích!)

Výsledek je v obou případech stejný. Spusťte obě následující buňky.

In [ ]:
import os, zipfile

def _ok(f):
    return os.path.exists(f) and os.path.getsize(f) > 1000

REZIM = "predpocitana"
for fname in ["train.npz", "val.npz", "test_features.npz", "samples.zip"]:
    if not _ok(fname):
        os.system(f"wget -q {DATA_BASE_URL}/{fname} -O {fname}")
    if not _ok(fname):                       # rychlá cesta nedostupná (např. privátní repo)
        if os.path.exists(fname):
            os.remove(fname)
        REZIM = "sobestacna"
        break

if REZIM == "predpocitana" and _ok("samples.zip") and not os.path.isdir("samples"):
    with zipfile.ZipFile("samples.zip") as z:
        z.extractall(".")     # zip už obsahuje prefix samples/
print("Režim dat:", REZIM)

In [ ]:
# Tahle buňka se spustí, jen když nejsou předpočítaná data. Pak si je notebook obstará sám.
# (Nemusíte jí rozumět — jen jednou připraví stejné soubory jako rychlá cesta.)
if REZIM == "sobestacna":
    import glob, shutil, urllib.request
    import numpy as np
    import torch
    import torch.nn as nn
    from PIL import Image
    from torch.utils.data import DataLoader, Dataset
    from torchvision import transforms
    from torchvision.models import resnet50, ResNet50_Weights
    from sklearn.model_selection import train_test_split

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    if dev == "cpu":
        print("⚠️  GPU NENÍ zapnuté → Runtime → Change runtime type → GPU. Na CPU to potrvá ~20 min.")

    URL = "https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip"
    if not os.path.exists("cell_images.zip"):
        print("stahuji obrázky (~350 MB)…")
        urllib.request.urlretrieve(URL, "cell_images.zip")
    if not os.path.isdir("cell_images"):
        with zipfile.ZipFile("cell_images.zip") as z:
            z.extractall(".")

    items = []
    for p in glob.glob("**/*.png", recursive=True):
        par = os.path.basename(os.path.dirname(p)).lower()
        if "parasitized" in par:
            items.append((p, 1))
        elif "uninfected" in par:
            items.append((p, 0))
    items.sort(key=lambda t: (os.path.basename(t[0]), t[1]))   # deterministické pořadí
    print("obrázků:", len(items))

    tfm = transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

    class _DS(Dataset):
        def __init__(self, it): self.it = it
        def __len__(self): return len(self.it)
        def __getitem__(self, i):
            p, yy = self.it[i]
            return tfm(Image.open(p).convert("RGB")), yy

    enc = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    enc.fc = nn.Identity(); enc.eval().to(dev)
    feats, labs = [], []
    with torch.no_grad():
        for k, (xb, yb) in enumerate(DataLoader(_DS(items), batch_size=128, num_workers=2)):
            feats.append(enc(xb.to(dev)).cpu().numpy().astype(np.float16))
            labs.append(np.asarray(yb))
            if k % 20 == 0:
                print(f"  zpracováno {k*128}/{len(items)}")
    Xall = np.concatenate(feats).astype(np.float16)
    yall = np.concatenate(labs).astype(np.int64)

    Xtmp, Xte, ytmp, yte = train_test_split(Xall, yall, test_size=0.15,
                                            stratify=yall, random_state=42)
    Xtr_, Xva_, ytr_, yva_ = train_test_split(Xtmp, ytmp, test_size=0.1765,
                                              stratify=ytmp, random_state=42)
    np.savez_compressed("train.npz", X=Xtr_, y=ytr_)
    np.savez_compressed("val.npz", X=Xva_, y=yva_)
    np.savez_compressed("test_features.npz", X=Xte)   # bez labelů (soutěž)

    os.makedirs("samples/Parasitized", exist_ok=True)
    os.makedirs("samples/Uninfected", exist_ok=True)
    for p in [p for p, l in items if l == 1][:15]:
        shutil.copy(p, "samples/Parasitized/")
    for p in [p for p, l in items if l == 0][:15]:
        shutil.copy(p, "samples/Uninfected/")
    print("hotovo — data připravena.")

## 2 · Podívejme se na buňky

Než začneme cokoli trénovat, podívejme se vlastníma očima, co má model rozlišit.
U **napadených** buněk bývá vidět malá fialová tečka — to je parazit *Plasmodium* obarvený
při přípravě nátěru. **Zdravé** buňky jsou rovnoměrně růžové.

In [ ]:
import glob
from PIL import Image

paras = sorted(glob.glob("samples/**/*arasitized*/*.png", recursive=True)) \
        or sorted(glob.glob("samples/**/Parasitized*/*.png", recursive=True))
healthy = sorted(glob.glob("samples/**/*ninfected*/*.png", recursive=True)) \
        or sorted(glob.glob("samples/**/Uninfected*/*.png", recursive=True))

fig, axes = plt.subplots(2, 6, figsize=(13, 4.5))
for j in range(6):
    if j < len(paras):
        axes[0, j].imshow(Image.open(paras[j]));
    axes[0, j].set_title("napadená", color="crimson", fontsize=10); axes[0, j].axis("off")
    if j < len(healthy):
        axes[1, j].imshow(Image.open(healthy[j]))
    axes[1, j].set_title("zdravá", color="seagreen", fontsize=10); axes[1, j].axis("off")
plt.suptitle("Vidíte u napadených buněk tu fialovou tečku (parazita)?", fontsize=12)
plt.tight_layout(); plt.show()

## 3 · Co je *encoder* a *featury*

Lidské oko parazita pozná. Jak ho ale „uvidí" počítač, který zná jen čísla?

Tady přichází na řadu **ResNet** — náš *encoder*. Je to konvoluční síť předtrénovaná na
milionech fotek. Když jí předhodíme obrázek buňky, **promění ho na vektor 2048 čísel**.
Tahle čísla shrnují, *co na obrázku je* — jaké tam jsou tvary, textury, skvrny. Říká se jim
**featury** (příznaky).

Klíčová myšlenka:

> Encoder jsme **zmrazili** — neučí se. Jen překládá `obrázek → 2048 čísel`.
> My pak učíme jen malou hlavu, která z těch 2048 čísel rozhodne *infikovaná / zdravá*.

Načteme si připravené featury a podíváme se, jak vypadají.

In [ ]:
train = np.load("train.npz")
val   = np.load("val.npz")

X_train, y_train = train["X"].astype(np.float32), train["y"].astype(np.int64)
X_val,   y_val   = val["X"].astype(np.float32),   val["y"].astype(np.int64)

print("trénink:", X_train.shape, " (počet buněk, počet featur)")
print("ověření:", X_val.shape)
print("\nKaždá buňka =", X_train.shape[1], "čísel.")
print("Vyváženost tříd v tréninku:",
      "zdravých", int((y_train == 0).sum()), "| napadených", int((y_train == 1).sum()))
print("\nPrvních 8 featur první buňky:", np.round(X_train[0, :8], 2))

### Magický okamžik: featury už třídy rozdělují

2048 rozměrů si nedokážeme představit. Promítneme je proto pomocí metody **PCA** do roviny
(2 rozměry) a obarvíme body podle skutečné třídy. Pokud encoder odvedl dobrou práci, uvidíme
**dva oddělené shluky** — ještě **předtím**, než jsme cokoli natrénovali. To je důkaz, že
předtrénovaná síť už „těžkou práci vidění" udělala za nás a nám zbývá jen najít dělící čáru.

In [ ]:
from sklearn.decomposition import PCA

idx = np.random.choice(len(X_train), size=3000, replace=False)  # podvzorek kvůli rychlosti
pts = PCA(n_components=2).fit_transform(X_train[idx])

plt.figure(figsize=(7, 6))
for label, name, col in [(0, "zdravá", "seagreen"), (1, "napadená", "crimson")]:
    m = y_train[idx] == label
    plt.scatter(pts[m, 0], pts[m, 1], s=6, alpha=0.4, label=name, color=col)
plt.legend(); plt.title("PCA featur — buňky se samy rozdělily do dvou shluků")
plt.xlabel("hlavní komponenta 1"); plt.ylabel("hlavní komponenta 2"); plt.show()

## 4 · Standardizace featur

Drobný, ale užitečný krok. Různé featury mají různě velké hodnoty. Když je všechny převedeme
na společné měřítko (průměr 0, rozptyl 1), neuronová síť se učí rychleji a stabilněji.

> ⚠️ Důležité pravidlo: měřítko spočítáme **jen z trénovacích dat** (`fit`) a stejné měřítko
> pak použijeme na ověřovací i testovací data (`transform`). Jinak bychom „nakoukli" do dat,
> která máme předstírat, že neznáme.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit JEN na tréninku
X_val_s   = scaler.transform(X_val)

# převedeme na tensory pro PyTorch
Xtr = torch.tensor(X_train_s, dtype=torch.float32, device=device)
ytr = torch.tensor(y_train,   dtype=torch.float32, device=device)
Xva = torch.tensor(X_val_s,   dtype=torch.float32, device=device)
print("připraveno:", Xtr.shape, "na zařízení", device)

## 5 · Vaše vlastní klasifikační hlava 🔬

Teď to nejzábavnější — **navrhnete vlastní malou neuronovou síť**. Dostane na vstupu 2048
featur a na výstupu vydá jediné číslo (*logit*), které po převedení funkcí *sigmoid* říká
**pravděpodobnost, že je buňka napadená**.

Baseline níže má jednu skrytou vrstvu. **Tady experimentujte:**
- `hidden` — kolik neuronů má skrytá vrstva (víc = větší kapacita, ale i riziko přeučení),
- `dropout` — náhodně „vypne" část neuronů při tréninku, brání přeučení (0 až ~0.5),
- můžete přidat **další vrstvy** (`nn.Linear`, `nn.ReLU`, `nn.Dropout`).

In [ ]:
class Hlava(nn.Module):
    def __init__(self, in_dim=2048, hidden=128, dropout=0.3):   # 🔬 EXPERIMENT: hidden, dropout
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),       # 🔬 EXPERIMENT: zkus přidat další vrstvy nad tenhle řádek
        )

    def forward(self, x):
        return self.net(x).squeeze(1)   # vrací logit, tvar (batch,)


model = Hlava(in_dim=Xtr.shape[1], hidden=128, dropout=0.3).to(device)
print(model)
print("počet trénovaných vah:", sum(p.numel() for p in model.parameters()))

## 6 · Trénink 🔬

Trénink znamená: model hádá, porovnáme jeho odhad se správnou odpovědí, spočítáme **chybu**
(*loss*) a malými krůčky vahy upravíme, aby chyba klesala. Jeden průchod všemi daty = jedna
**epocha**.

Páky, se kterými si můžete hrát:
- `EPOCHS` — kolikrát projdeme data (víc se naučí, ale může se přeučit),
- `LR` (learning rate) — velikost krůčku při učení,
- `WEIGHT_DECAY` — mírná penalizace velkých vah, brání přeučení,
- `POS_WEIGHT` — **jak draho stojí přehlédnutý nemocný**. Hodnota > 1 nutí model brát napadené
  buňky vážněji (zvýší senzitivitu na úkor falešných poplachů). To je přímo páka na medicínský
  kompromis, o kterém bude řeč v sekci 7!

In [ ]:
# ====== 🔬 EXPERIMENT: hyperparametry ======
EPOCHS       = 25
BATCH        = 256
LR           = 1e-3
WEIGHT_DECAY = 1e-4
POS_WEIGHT   = 1.0      # > 1 = přísnější na přehlédnuté nemocné
# ===========================================

loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT, device=device))
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

hist = {"train_loss": [], "val_acc": []}
n = Xtr.shape[0]
for epoch in range(EPOCHS):
    model.train()
    perm = torch.randperm(n, device=device)
    running = 0.0
    for i in range(0, n, BATCH):
        idx = perm[i:i + BATCH]
        opt.zero_grad()
        loss = loss_fn(model(Xtr[idx]), ytr[idx])
        loss.backward()
        opt.step()
        running += loss.item() * len(idx)
    # vyhodnocení na ověřovací sadě
    model.eval()
    with torch.no_grad():
        val_prob = torch.sigmoid(model(Xva)).cpu().numpy()
    acc = ((val_prob > 0.5).astype(int) == y_val).mean()
    hist["train_loss"].append(running / n)
    hist["val_acc"].append(acc)
    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(f"epocha {epoch:2d}  chyba={running/n:.4f}  přesnost na ověření={acc:.4f}")
print("\nHotovo. Nejlepší přesnost na ověření:", round(max(hist['val_acc']), 4))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist["train_loss"]); ax[0].set_title("Trénovací chyba (loss)")
ax[0].set_xlabel("epocha"); ax[0].set_ylabel("chyba")
ax[1].plot(hist["val_acc"], color="darkorange"); ax[1].set_title("Přesnost na ověření")
ax[1].set_xlabel("epocha"); ax[1].set_ylabel("přesnost"); ax[1].set_ylim(0, 1)
plt.tight_layout(); plt.show()

## 7 · Proč v medicíně nestačí „95 % přesnost" 🩺

Představte si model, který o **každé** buňce řekne „zdravá". Když je v nátěru 95 % buněk
zdravých, má tenhle hloupý model rovnou **95 % přesnost** — a přitom **nezachytí ani jednoho
nemocného**. Přesnost (accuracy) nás obelhala.

V medicíně rozlišujeme dva druhy chyb a každý má jinou cenu:

|  | model řekl **zdravá** | model řekl **napadená** |
|---|---|---|
| **opravdu zdravá** | ✅ správně negativní (TN) | ⚠️ **falešný poplach** (FP) |
| **opravdu napadená** | ☠️ **přehlédnutý nemocný** (FN) | ✅ správně pozitivní (TP) |

- **Senzitivita** = TP / (TP + FN) = *jaký podíl skutečně nemocných model zachytí.*
  Nízká senzitivita = přehlížíme nemocné. **Tohle může stát život.**
- **Specificita** = TN / (TN + FP) = *jaký podíl zdravých model správně propustí.*
  Nízká specificita = strašíme zdravé a zbytečně je posíláme na další vyšetření.

Spočítejme si je z **matice záměn**.

In [ ]:
from sklearn.metrics import confusion_matrix

def matice_a_metriky(y_true, y_prob, prah=0.5):
    y_pred = (y_prob >= prah).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    return (tn, fp, fn, tp), sens, spec

(tn, fp, fn, tp), sens, spec = matice_a_metriky(y_val, val_prob, prah=0.5)

cm = np.array([[tn, fp], [fn, tp]])
fig, ax = plt.subplots(figsize=(4.8, 4.2))
ax.imshow(cm, cmap="Blues")
for (r, c), v in np.ndenumerate(cm):
    ax.text(c, r, str(v), ha="center", va="center", fontsize=14)
ax.set_xticks([0, 1]); ax.set_xticklabels(["řekl zdravá", "řekl napadená"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["je zdravá", "je napadená"])
ax.set_title("Matice záměn (práh 0.5)"); plt.tight_layout(); plt.show()

print(f"senzitivita = {sens:.3f}  (podíl zachycených nemocných)")
print(f"specificita = {spec:.3f}  (podíl správně propuštěných zdravých)")
print(f"přehlédnutých nemocných (FN): {fn}   |   falešných poplachů (FP): {fp}")

### Práh rozhoduje o kompromisu

Model vydává **pravděpodobnost** 0–1. My musíme zvolit **práh**, nad kterým buňku prohlásíme
za napadenou. Posuneme-li práh **dolů**, zachytíme víc nemocných (↑ senzitivita), ale přibude
falešných poplachů (↓ specificita). A naopak. Celý ten kompromis zachycuje **ROC křivka**.

Naše **soutěžní metrika** je jeden konkrétní bod na téhle křivce:

> **Senzitivita při specificitě ≥ 95 %** — tedy: *kolik nemocných dokážeme zachytit, když si
> dovolíme nanejvýš 5 % falešných poplachů?* Čím vyšší, tím lepší model.

In [ ]:
from sklearn.metrics import roc_curve

def senzitivita_pri_specificite(y_true, y_prob, min_spec=0.95):
    fpr, tpr, thr = roc_curve(y_true, y_prob)          # fpr = 1 - specificita
    ok = fpr <= (1.0 - min_spec)
    best = int(np.argmax(np.where(ok, tpr, -1)))
    return tpr[best], thr[best], 1 - fpr[best]

fpr, tpr, thr = roc_curve(y_val, val_prob)
sens95, prah95, spec95 = senzitivita_pri_specificite(y_val, val_prob, 0.95)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, lw=2, label="ROC křivka")
plt.axvline(0.05, ls="--", color="gray", label="hranice 5 % falešných poplachů")
plt.scatter([1 - spec95], [sens95], color="crimson", zorder=5,
            label=f"náš bod: senzitivita={sens95:.3f}")
plt.plot([0, 1], [0, 1], ls=":", color="lightgray")
plt.xlabel("1 − specificita  (falešné poplachy)"); plt.ylabel("senzitivita")
plt.title("ROC křivka"); plt.legend(loc="lower right"); plt.show()

print(f"➡️  SOUTĚŽNÍ SKÓRE (na ověření): senzitivita @ spec≥95 % = {sens95:.3f}")
print(f"    (odpovídající práh = {prah95:.3f}, dosažená specificita = {spec95:.3f})")

### Vyzkoušejte si posuvník prahu

Posouvejte práh a sledujte, jak se mění matice záměn i obě metriky. Najdete práh, který drží
specificitu kolem 95 %?

In [ ]:
try:
    from ipywidgets import interact, FloatSlider

    def ukaz(prah=0.5):
        (tn, fp, fn, tp), s, sp = matice_a_metriky(y_val, val_prob, prah)
        print(f"práh={prah:.2f} | senzitivita={s:.3f} | specificita={sp:.3f} "
              f"| přehlédnutí(FN)={fn} | falešné poplachy(FP)={fp}")

    interact(ukaz, prah=FloatSlider(min=0.01, max=0.99, step=0.01, value=0.5))
except Exception as e:
    print("(posuvník nedostupný, ukážu tabulku)", e)
    for prah in [0.1, 0.3, 0.5, 0.7, 0.9]:
        (tn, fp, fn, tp), s, sp = matice_a_metriky(y_val, val_prob, prah)
        print(f"práh={prah:.2f} | senzitivita={s:.3f} | specificita={sp:.3f} "
              f"| FN={fn} | FP={fp}")

## 8 · Ladění pro soutěž 🔬🏆

Teď zpátky nahoru a vylepšujte! Cílem je co **nejvyšší senzitivita při specificitě ≥ 95 %**
na skrytém testu. Páky, které máte k dispozici:

| kde | páka | co zkusit |
|---|---|---|
| sekce 5 | architektura hlavy | větší `hidden`, jiný `dropout`, další vrstvy |
| sekce 6 | `EPOCHS` | víc epoch — ale pozor na přeučení (sledujte ověření) |
| sekce 6 | `LR` | např. 3e-4, 1e-3, 3e-3 |
| sekce 6 | `WEIGHT_DECAY` | větší hodnota brzdí přeučení |
| sekce 6 | `POS_WEIGHT` | > 1 zvýší senzitivitu (přísnější na nemocné) |
| sekce 4 | standardizace | už zapnutá — zkuste, jaký je rozdíl bez ní |

**Postup:** změňte jednu věc → spusťte sekce 5, 6, 7 → podívejte se na soutěžní skóre →
opakujte. Vždycky měňte **jen jednu věc**, ať víte, co pomohlo.

> 💡 Tip: nejde jen o přesnost. Když model dělá hodně falešných poplachů, „dojde mu rozpočet"
> 5 % specificity dřív a senzitivita spadne. Hledáte model, který si je svými předpověďmi
> **jistý ve správných případech**.

## 9 · Odevzdání do soutěže 📤

Až budete s modelem spokojení, vygenerujte odevzdání. Spočítáme pravděpodobnosti na **skrytém
testu** (u kterého neznáte správné odpovědi) a uložíme je do souboru `predikce_<TÝM>.csv`.
Ten odevzdejte školiteli — on spočítá vaše skóre na skrytých odpovědích a vyhlásí vítěze.

> Odevzdávají se **pravděpodobnosti**, ne hotová rozhodnutí — školitel sám najde práh pro
> specificitu 95 %. Vy se tedy nemusíte starat o volbu prahu, jen ať jsou vaše pravděpodobnosti
> co nejlépe seřazené (jisté tam, kde mají být).

In [ ]:
test = np.load("test_features.npz")
X_test_s = scaler.transform(test["X"].astype(np.float32))
Xte = torch.tensor(X_test_s, dtype=torch.float32, device=device)

model.eval()
with torch.no_grad():
    test_prob = torch.sigmoid(model(Xte)).cpu().numpy()

fname = f"predikce_{TEAM_NAME}.csv"
np.savetxt(fname, test_prob, fmt="%.6f", header="prob", comments="")
print(f"uloženo: {fname}  ({len(test_prob)} buněk)")
print("ukázka prvních 5 pravděpodobností:", np.round(test_prob[:5], 4))

# v Google Colabu rovnou nabídne stažení souboru:
try:
    from google.colab import files
    files.download(fname)
except Exception:
    print("(mimo Colab — soubor najdete vedle notebooku)")

## 🎓 Co jste se naučili

- **Transfer learning**: stačí dotrénovat malou hlavu nad zmraženým ResNetem — žádné
  superpočítače, žádné miliony obrázků.
- Featury jsou **2048 čísel**, do kterých předtrénovaná síť zhustila „co na obrázku je".
- **Přesnost klame.** V medicíně sledujeme **senzitivitu** (nepřehlédnout nemocného) a
  **specificitu** (nestrašit zdravé), čteme **matici záměn** a **ROC křivku**.
- **Práh** je rozhodnutí o ceně chyby — a ta cena není symetrická.

### Otázky k zamyšlení
1. Proč by se model nasazený v nemocnici mohl chovat hůř než tady na datasetu? (Jiný mikroskop,
   jiné barvení, jiná populace pacientů…)
2. Měla by konečné rozhodnutí dělat AI sama, nebo jen *upozorňovat* lékaře? Kde nastavit práh?
3. Co by se stalo se senzitivitou, kdyby byla v reálu napadená jen **1 %** buněk?

**Gratulace — mezi vaším notebookem a reálným dopadem na zdravotnictví není tak daleko. 🦟➡️🩺**